# Лекция 4. Безусловная оптимизация: условия оптимальности, градиентный спуск и скорость сходимости

**Курс:** Вычислительная оптимизация · **Часть II. Безусловная оптимизация и ньютоновские методы** · Лекция 4 из 16 · 30 сентября 2026

**Материалы:** [конспект-ноутбук](lecture04.ipynb) · [демо-ноутбук](demo04.ipynb) · [разбор упражнений](exercises04.ipynb) · [сценарий доски](board04.md)

---

Часть I закончилась на том, как *выглядит* решение: $\nabla f=0$ для выпуклых задач без ограничений, баланс сил $\nabla f=\sum\mu_i\nabla h_i$ у стенок. Часть II — о том, как решение *находить*. Сегодня ограничений нет вовсе, $\Omega=\mathbb{R}^n$, и три вопроса:

1. Как узнать точку минимума, если функция не выпукла — что даёт $\nabla f=0$ и что добавляет гессиан?
2. Как к минимуму *идти* — куда шагать и на сколько?
3. Почему один и тот же метод на одной задаче делает 18 шагов, а на другой — 13 тысяч, и что с этим делать?

Ответ на третий вопрос — одно число, **число обусловленности** $\kappa$. Оно будет главным героем лекции.

## 1. Повторение и мост: от «как выглядит решение» к «как его найти»

### 1.1. Что уже умеем

| Вопрос | Ответ | Где |
|---|---|---|
| Есть ли минимум? | Да, если $f$ непрерывна и растёт на бесконечности ($f\to\infty$ при $\lVert x\rVert\to\infty$): множество $\{f\le f(x_0)\}$ ограничено, минимум ищется на компакте | Л1, §5 |
| Как проверить точку в выпуклой задаче? | $\nabla f(x^\ast)=0$ — необходимо и достаточно, и минимум глобальный. Для невыпуклых — «лишь необходимое условие, об этом лекция 4» | Л2, §7 |
| Что такое «решить численно»? | Итерации $x_{k+1}=x_k+p_k$, остановка при $\lVert\nabla f(x_k)\rVert\le\varepsilon$; важны число итераций и цена одной | Л1, §8 |

### 1.2. Три задачи на сегодня

<img src="img/00_hooks.png" width="900" alt="три безусловные задачи: функция Розенброка, цепь без пола, логистическая регрессия">

**Розенброк.** $f(x)=(1-x_1)^2+100\,(x_2-x_1^2)^2$ — «банановая долина» из ДЗ 2 (задача 2а). Вы уже знаете: функция невыпукла, стационарная точка одна — $(1,1)$, и это глобальный минимум. Осталось до него дойти. Это стандартный полигон для методов части II: изогнутая узкая долина, на дне которой методы ползут.

**Цепь без пола** (лекция 1, §7.3). $40$ грузов на пружинах, $80$ переменных, минимум энергии — выпуклая квадратичная задача. Без ограничений её решает `np.linalg.solve`, но сегодня мы запустим на ней градиентный спуск и увидим, что даже выпуклая квадратичная задача может быть *трудной*.

**Логистическая регрессия** (лекция 1, §7.5). По возрасту и стажу предсказать класс: найти $w$, минимизирующий $\frac1n\sum_i\log(1+e^{-y_i x_i^\top w})$. Выпуклая, гладкая, без ограничений — и на ней градиентный спуск без одной простой подготовки не сходится вовсе.

Во всех трёх задачах нет ни одной стенки. Вспомним систему условий оптимальности лекции 3: $\nabla_xL=0$, допустимость, $\mu\ge0$, $\mu_ih_i=0$. Нет ограничений — нет множителей, и вся система сводится к одному уравнению $\nabla f(x^\ast)=0$. **Нет стенок — нет сил.**

**Что мы знаем о множителях и чего пока нет.** После лекции 3 у множителя два прочтения — сила реакции стенки и цена ограничения, — а система ККТ — сертификат: подставили решение и множители, все нули — решение доказано. Чего мы *не* умеем: находить множители, не угадывая заранее, какие стенки активны, и получать гарантированные оценки $f^\ast$ снизу. Это даёт **двойственность Лагранжа** — лекция 10; кому не терпится, черновик лежит в [`lectures/extra/duality.md`](../extra/duality.md). Задача 4 в ДЗ 2 продолжает тему множителей уже сейчас.

### 1.3. Что сегодня

Условия оптимальности первого и второго порядка (разделы 2–3) → градиентный спуск: направление, шаг, лемма о спуске (4) → число обусловленности и скорость (5–6) → один шаг Ньютона как тизер лекции 5 (7). Семинар — задачи у доски (11–12).

## 2. Стационарные точки: условие первого порядка

**Теорема (необходимое условие первого порядка).** Если $x^\ast$ — локальный минимум $f\in C^1$ на $\mathbb{R}^n$, то $\nabla f(x^\ast)=0$.

*Доказательство.* Пусть $g=\nabla f(x^\ast)\ne0$. Пойдём против градиента: по Тейлору $f(x^\ast-tg)=f(x^\ast)-t\lVert g\rVert^2+o(t)$, и при малом $t>0$ значение меньше $f(x^\ast)$ — сколь угодно близко к $x^\ast$ есть точки лучше. Противоречие с локальностью минимума. $\square$

Точки с $\nabla f=0$ называются **стационарными**. Теорема говорит только «минимум — стационарная точка», но не наоборот: стационарной может быть и точка максимума, и **седло** — точка, из которой в одни стороны функция растёт, а в другие убывает.

<img src="img/01_stationary.png" width="900" alt="минимум, максимум и седло: линии уровня и поле антиградиентов">

Стрелки — антиградиент $-\nabla f$, то есть направление, куда пойдёт спуск. В минимуме стрелки сходятся, в максимуме расходятся, в седле — сходятся вдоль одной оси и расходятся вдоль другой. Для методов часть II это важнее, чем кажется: градиентный спуск ищет точку, где стрелки исчезают, и седло для него — такая же стационарная точка, как минимум (упражнение 12.1). Спасает только то, что седло **неустойчиво**: любое отклонение с «хребта» уводит вниз.

**Обозначения на всю часть II.** $x^\ast$ — минимум, $e_k=x_k-x^\ast$ — ошибка $k$-й итерации, $g_k=\nabla f(x_k)$, $H(x)=\nabla^2f(x)$ — гессиан, симметричная матрица $n\times n$.

## 3. Второй порядок: необходимое, достаточное и зазор между ними

Раз первого порядка мало, привлекаем гессиан. По Тейлору второго порядка вдоль направления $p$:

$$
f(x^\ast+tp)=f(x^\ast)+t\,\nabla f(x^\ast)^\top p+\tfrac12t^2\,p^\top\nabla^2f(x^\ast)\,p+o(t^2).
$$

В стационарной точке средний член нуль, и всё решает знак квадратичной формы.

**Теорема (второй порядок).** Пусть $f\in C^2$.

- *Необходимое условие:* если $x^\ast$ — локальный минимум, то $\nabla f(x^\ast)=0$ и $\nabla^2f(x^\ast)\succeq0$.
- *Достаточное условие:* если $\nabla f(x^\ast)=0$ и $\nabla^2f(x^\ast)\succ0$, то $x^\ast$ — **строгий** локальный минимум.

*Доказательство.* Необходимость: будь $p^\top Hp<0$ для какого-то $p$, вдоль $p$ функция убывала бы при малых $t$ — снова точки лучше рядом. Достаточность: $p^\top Hp\ge\lambda_{\min}\lVert p\rVert^2>0$, и при малых $\lVert tp\rVert$ квадратичный член перевешивает остаток $o(t^2)$, так что $f(x^\ast+tp)>f(x^\ast)$ для всех $p\ne0$. $\square$

**Зазор.** Между $\succeq0$ и $\succ0$ есть щель, и в ней условия молчат. Три функции одной переменной — $x^4$, $x^3$, $-x^4$ — имеют в нуле одинаковые $f'(0)=f''(0)=0$, а судьбы разные: минимум, не экстремум, максимум. Различить их могут только старшие производные, и общего рецепта нет; в двух переменных то же: $x_1^2+x_2^4$ и $x_1^2-x_2^4$ имеют один гессиан $\operatorname{diag}(2,0)$ (упражнение 12.2).

<img src="img/02_second_order.png" width="900" alt="зазор между условиями: x^4, x^3, -x^4; четыре квадратичных ландшафта">

**Квадратичный случай — без зазора.** Для $f(x)=\tfrac12x^\top Qx+c^\top x$ гессиан постоянен и равен $Q$, остатка Тейлора нет, и ответ полный:

| $Q$ | Ландшафт | Стационарные точки |
|---|---|---|
| $Q\succ0$ | чаша | одна, $x^\ast=-Q^{-1}c$ — глобальный минимум (лекция 2, §7) |
| $Q$ индефинитна | седло | стационарная точка есть, минимума нет: вдоль собственного вектора с $\lambda<0$ функция уходит в $-\infty$ |
| $Q\succeq0$ вырождена, $c\in\operatorname{range}Q$ | жёлоб | целая прямая (плоскость) минимумов — как в МНК с линейно зависимыми столбцами |
| $Q\succeq0$ вырождена, $c\notin\operatorname{range}Q$ | наклонный жёлоб | стационарных точек нет, $f\to-\infty$ вдоль жёлоба |

Эта таблица понадобится в лекции 5: метод Ньютона на каждом шаге минимизирует именно такую модель, и «седло» в таблице означает, что шаг Ньютона может пойти *вверх*.

**Выпуклость закрывает зазор.** Если $f$ выпукла, $\nabla f(x^\ast)=0$ — необходимо и достаточно, и минимум глобальный (лекция 2, §7): гессиан здесь не нужен вообще. Гессиан нужен ровно тогда, когда выпуклости нет.

**Розенброк.** Из ДЗ 2: единственная стационарная точка $(1,1)$, в ней

$$
\nabla^2f(1,1)=\begin{pmatrix}802&-400\\-400&200\end{pmatrix},\qquad \lambda_{\min}\approx0.40,\quad\lambda_{\max}\approx1001.6 .
$$

Оба собственных числа положительны — по достаточному условию это строгий локальный минимум, а так как $f\ge0=f(1,1)$, ещё и глобальный. Запомните эти два числа: их отношение $\kappa\approx2508$ объяснит всё, что случится с градиентным спуском в разделе 5.

## 4. Градиентный спуск: направление, шаг, лемма о спуске

Условия найдены; теперь — как к ним прийти. Итерационный метод из лекции 1 (§8): $x_{k+1}=x_k+p_k$. Нужно выбрать направление $p_k$ и длину шага.

**Направление.** Производная $f$ в точке $x$ по единичному направлению $p$ равна $\nabla f(x)^\top p$. По неравенству Коши–Буняковского $\nabla f^\top p\ge-\lVert\nabla f\rVert$, и равенство достигается при $p=-\nabla f/\lVert\nabla f\rVert$: **антиградиент — направление самого крутого спуска** (упражнение 12.8). Все направления с $p^\top\nabla f<0$ — **направления спуска**: они образуют полуплоскость (полупространство), а её граница — касательная к линии уровня, ведь $\nabla f$ ортогонален линии уровня.

<img src="img/03_descent_direction.png" width="900" alt="направления спуска и функция одного шага φ(α) с тремя способами выбрать длину">

**Метод.** Градиентный спуск — это

$$
\boxed{\;x_{k+1}=x_k-\alpha_k\nabla f(x_k)\;}
$$

и вся содержательная часть — в выборе $\alpha_k$. Вдоль луча $x_k-\alpha\nabla f(x_k)$ функция становится функцией одной переменной $\varphi(\alpha)=f(x_k-\alpha\nabla f(x_k))$, и $\varphi'(0)=-\lVert\nabla f(x_k)\rVert^2<0$: при малых $\alpha$ спуск гарантирован. Вопрос — насколько малых.

**Постоянный шаг и лемма о спуске.** Пусть градиент липшицев: $\lVert\nabla f(x)-\nabla f(y)\rVert\le L\lVert x-y\rVert$; для $C^2$-функции это значит $\nabla^2f\preceq L I$, а для квадратичной $L=\lambda_{\max}(Q)$. Тогда квадратичная модель с кривизной $L$ — **верхняя оценка** функции (упражнение 12.6):

$$
f(y)\le f(x)+\nabla f(x)^\top(y-x)+\tfrac L2\lVert y-x\rVert^2,
\qquad\text{откуда}\qquad
f(x-\alpha\nabla f)\le f(x)-\alpha\Big(1-\frac{\alpha L}2\Big)\lVert\nabla f\rVert^2 .
$$

При $0<\alpha<2/L$ скобка положительна, и функция гарантированно убывает; при $\alpha=1/L$ убывание наибольшее, $f$ уменьшается хотя бы на $\lVert\nabla f\rVert^2/(2L)$. Суммируя по итерациям: $\sum_k\lVert\nabla f(x_k)\rVert^2\le2L\,(f(x_0)-f^\ast)<\infty$, значит $\nabla f(x_k)\to0$ — метод сходится к стационарной точке, для выпуклой функции — к глобальному минимуму. Граница $2/L$ настоящая: за ней шаг перепрыгивает дно и метод не сходится (раздел 5, Розенброк).

**Точный шаг.** Минимизировать $\varphi(\alpha)$ по $\alpha$. Для квадратичной $f=\tfrac12x^\top Qx+c^\top x$ это делается явно: с $g=\nabla f(x_k)$

$$
\alpha_k^\ast=\frac{g^\top g}{g^\top Qg}.
$$

Для общей $f$ точный поиск — сама по себе задача оптимизации, и на практике его заменяют приближённым.

**Backtracking — рецепт на сегодня.** Начать с $\alpha=1$ и делить пополам, пока не выполнено **условие Армихо** — «убыло хотя бы малую долю того, что обещала линейная модель»:

```python
alpha = 1.0
while f(x - alpha * g) > f(x) - 1e-4 * alpha * (g @ g):
    alpha /= 2
x = x - alpha * g
```

Три строки, работают без знания $L$ и на невыпуклых функциях. Почему это сходится, чем плоха константа $10^{-4}$ и что такое условие Вольфе — лекция 6; на картинке выше видно главное: backtracking останавливается недалеко от точного шага.

**Остановка.** Как в лекции 1: $\lVert\nabla f(x_k)\rVert\le\varepsilon$. Для выпуклой $f$ малый градиент означает близость к глобальному минимуму, для невыпуклой — к стационарной точке; какой именно, метод не знает.

## 5. Скорость: число обусловленности

Метод сходится. Но *как быстро*? Ответ виден на квадратичной функции, а к общей переносится почти дословно.

**Квадратичная задача в собственных осях.** Пусть $f=\tfrac12x^\top Qx$, $Q\succ0$ с собственными числами $0<\lambda_1\le\dots\le\lambda_n$; минимум в нуле, ошибка $e_k=x_k$. Шаг $x_{k+1}=x_k-\alpha Qx_k=(I-\alpha Q)x_k$: в базисе собственных векторов каждая компонента ошибки умножается на свой множитель $1-\alpha\lambda_i$. Чтобы сходились все, нужно $|1-\alpha\lambda_i|<1$ для всех $i$, то есть $0<\alpha<2/\lambda_{\max}$ — снова граница $2/L$. Возьмём $\alpha=1/\lambda_{\max}$: компонента вдоль $\lambda_{\max}$ обнуляется за один шаг, а самая медленная, вдоль $\lambda_{\min}$, сжимается всего в

$$
1-\frac{\lambda_{\min}}{\lambda_{\max}}=1-\frac1\kappa,\qquad
\boxed{\;\kappa=\frac{\lambda_{\max}(Q)}{\lambda_{\min}(Q)}\;}
$$

— **число обусловленности**. За $k$ шагов ошибка умножается на $(1-1/\kappa)^k\approx e^{-k/\kappa}$, и чтобы уменьшить её в $1/\varepsilon$ раз, нужно

$$
k\approx\kappa\ln\frac1\varepsilon\quad\text{итераций.}
$$

Число итераций **пропорционально $\kappa$**. При $\kappa=2$ одна верная цифра стоит $3$ итерации, при $\kappa=50$ — $114$; лучший постоянный шаг $2/(\lambda_{\max}+\lambda_{\min})$ улучшает множитель до $(\kappa-1)/(\kappa+1)$ — вдвое, не больше (упражнение 12.3). Точный шаг не спасает: для него $f$ сжимается в $\big(\frac{\kappa-1}{\kappa+1}\big)^2$ за итерацию (теорема Канторовича), и итераций по-прежнему порядка $\kappa$.

<img src="img/04_zigzag.png" width="900" alt="зигзаг градиентного спуска с точным шагом при κ=2 и κ=50">

**Зигзаг.** При точном шаге соседние градиенты ортогональны (упражнение 12.8в), и на вытянутом эллипсе метод мечется поперёк долины, продвигаясь вдоль неё чуть-чуть за шаг: $18$ итераций при $\kappa=2$ против $567$ при $\kappa=50$. Геометрия та же, что у Розенброка, только у него долина ещё и изогнута.

**Общий случай.** Если $mI\preceq\nabla^2f(x)\preceq LI$ всюду (сильно выпуклая функция с липшицевым градиентом), то градиентный спуск с шагом $1/L$ даёт $f(x_k)-f^\ast\le(1-m/L)^k\,(f(x_0)-f^\ast)$ — то же самое с $\kappa=L/m$. Для невыпуклой функции оценка локальная: вблизи минимума $\kappa=\kappa(\nabla^2f(x^\ast))$.

**Числа.**

- *Розенброк:* $\kappa(\nabla^2f(1,1))\approx2508$. Постоянный шаг $0.001$ (это $\approx1/\lambda_{\max}$): $32\,076$ итераций из $(-1.2,1)$ до $\lVert\nabla f\rVert\le10^{-6}$ — оценка $\kappa\ln10^6\approx3.5\cdot10^4$ попала. Шаг $0.0019$: $16\,851$. Шаг $0.002$ — чуть больше $2/\lambda_{\max}$: метод *не сходится*. Он даже подходит к минимуму вплотную, но минимум для такого шага неустойчив, и итерации раскачиваются, пока не осядут в колебании поперёк долины — $f$ прыгает между $\approx1.25\cdot10^{-3}$ и $1.26\cdot10^{-3}$ и не убывает. Backtracking: $13\,756$ итераций — вдвое лучше постоянного шага, потому что вдали от минимума долина шире и шаги крупнее. Всё это — демо, часть (c).
- *Цепь:* гессиан энергии — блочная матрица жёсткости $\operatorname{diag}(DK,DK)$ с трёхдиагональной $K$, её $\kappa\approx\big(\frac{2(N+1)}\pi\big)^2$: $49$ при $N=10$, $681$ при $N=40$, $2659$ при $N=80$ (упражнение 12.9). Выпуклая квадратичная задача, а градиентному спуску нужно $10\,575$ итераций при $N=40$ — и в $4$ раза больше при удвоении $N$. Чем длиннее цепь, тем «мягче» её самая медленная мода — тем хуже обусловленность (демо, часть e).
- *Логистическая регрессия:* признаки «возраст $\approx40\pm10$ лет» и «стаж $\approx5\pm1$ лет» плюс столбец единиц. Гессиан $\frac1nX^\top SX$ — взвешенная матрица вторых моментов признаков, и его $\kappa\approx7.5\cdot10^4$: градиентный спуск не достигает $\lVert\nabla f\rVert\le10^{-6}$ и за $20\,000$ итераций (демо, часть d).

**Масштабирование переменных — первое лекарство.** Число обусловленности — свойство не задачи, а *координат*. Заменим $u_2=5x_2$ в $f=\tfrac12(x_1^2+25x_2^2)$: эллипсы становятся кругами, $\kappa=1$, и градиентный спуск попадает в минимум за один шаг вместо $378$.

<img src="img/06_scaling.png" width="800" alt="масштабирование переменных: эллипсы превращаются в круги">

То же для логистической регрессии: стандартизуем признаки (вычесть среднее, поделить на разброс) — $\kappa$ падает с $7.5\cdot10^4$ до $2.75$, и метод сходится за $195$ итераций. Общее правило: шаг $x_{k+1}=x_k-\alpha D^{-1}\nabla f(x_k)$ с диагональной $D$ — это градиентный спуск в масштабированных переменных $u=D^{1/2}x$; выбор $D$ называют **предобуславливанием**. Простейший вариант — привести все переменные к величинам порядка единицы: расстояние в метрах и в километрах — это одна задача, но $\kappa$ у них отличается в $10^6$ раз (упражнение 12.5).

**Почему градиентный спуск всё же живёт.** Одна итерация стоит один градиент — $O(n)$ операций и памяти; для логистической регрессии с миллионом признаков ничего дороже мы себе позволить не можем, и $\kappa$ приходится чинить масштабированием, а не сменой метода. Там, где $n$ невелико, есть методы, которым $\kappa$ безразлично, — раздел 7.

## 6. Линейная, сверхлинейная, квадратичная

Мы сказали «ошибка сжимается в $q$ раз за шаг». Это — **линейная** сходимость, и не единственно возможная. Пусть $x_k\to x^\ast$.

| Тип | Определение | Что видно на графике $\log\lVert e_k\rVert$ | Верные цифры |
|---|---|---|---|
| линейная | $\lVert e_{k+1}\rVert\le q\lVert e_k\rVert$, $q<1$ | прямая с наклоном $\log q$ | по одной каждые $\ln10/\ln(1/q)$ итераций |
| сверхлинейная | $\lVert e_{k+1}\rVert/\lVert e_k\rVert\to0$ | всё круче и круче | всё больше за шаг |
| квадратичная | $\lVert e_{k+1}\rVert\le C\lVert e_k\rVert^2$ | «обрыв» | число цифр удваивается: $10^{-2},10^{-4},10^{-8},10^{-16}$ |

<img src="img/05_rates.png" width="900" alt="три скорости сходимости и измеренная линейная сходимость градиентного спуска">

Градиентный спуск — линейный с $q=1-1/\kappa$: на правой картинке ошибка на квадратичной задаче ложится на прямую, и наклон точно $\log_{10}(1-1/\kappa)$. Число обусловленности задаёт наклон, и ничем, кроме масштабирования, его не улучшить. Квадратичная сходимость — это когда из $10^{-2}$ в $10^{-16}$ попадают за три шага; такую даёт метод Ньютона.

## 7. Тизер: один шаг Ньютона

Градиентный спуск использует линейную модель функции. Возьмём квадратичную — Тейлора второго порядка в текущей точке:

$$
m_k(p)=f(x_k)+\nabla f(x_k)^\top p+\tfrac12\,p^\top\nabla^2f(x_k)\,p,
\qquad
\nabla m_k=0\ \Longrightarrow\ \boxed{\;p_k=-\nabla^2f(x_k)^{-1}\nabla f(x_k)\;}
$$

— **шаг Ньютона**. Если $f$ квадратична, модель точна, и шаг попадает в минимум *за одну итерацию* — это ровно $x^\ast=-Q^{-1}c$ из таблицы раздела 3; для цепи это `np.linalg.solve(H, -g)`, одна линейная система вместо $10\,575$ итераций. Если $f$ не квадратична, модель точна лишь вблизи, но там — квадратичная сходимость.

<img src="img/07_newton_teaser.png" width="900" alt="Розенброк: путь градиентного спуска и метода Ньютона, ошибки в логарифмической шкале">

На Розенброке из $(-1.2,1)$: Ньютон — $7$ итераций, градиентный спуск — $13\,756$. Ошибка Ньютона: $2.2,\ 2.2,\ 4.2,\ 0.48,\ 0.056,\ 10^{-5},\ 10^{-11}$ — сначала два шага почти впустую, на втором даже *хуже* (модель вдали от минимума врёт, и шаг улетает в $(0.77,-3.2)$), зато последние три — удвоение цифр. Числу обусловленности Ньютон безразличен: он делает ту же самую линейную замену координат, что мы делали руками в разделе 5, только автоматически и на каждом шаге — для логистической регрессии он сходится за $6$ итераций что с масштабированием признаков, что без.

Цена: гессиан ($n^2$ чисел) и линейная система ($n^3$ операций), а главное — вдали от минимума шаг может пойти не туда: если $\nabla^2f$ индефинитен, модель — седло из раздела 3, и её «минимума» не существует. Как это чинится (квази-Ньютон, линейный поиск, доверительная область) — лекции 5 и 6.

## 8. Что это даёт численно

**Диагноз по графику.** Постройте $\log\lVert\nabla f(x_k)\rVert$ или $\log(f(x_k)-f^\ast)$ по итерациям. Прямая с пологим наклоном — линейная сходимость с большим $\kappa$: проверьте масштабы переменных, прежде чем менять метод. Обрыв — ньютоновская сходимость: вы в области, где квадратичная модель точна.

**Стационарная точка — ещё не минимум.** Для невыпуклой $f$ малый градиент не различает минимум и седло. Если есть сомнения — собственные числа гессиана в найденной точке (демо, часть a); отрицательное собственное число указывает направление, вдоль которого можно спуститься ещё.

**Что делает `minimize` по умолчанию.** Без ограничений SciPy запускает BFGS — квази-ньютоновский метод: он не считает гессиан, а восстанавливает его по градиентам, и сходится сверхлинейно. Это тема лекции 5; после неё вы напишете градиентный спуск, Ньютон и BFGS сами и сравните их на Розенброке, цепи и регрессии (ДЗ 3).

**Откуда брать градиент.** В демо все градиенты выписаны руками. Что делать, когда $f$ — тысяча строк кода, — лекция 7: конечные разности и их погрешность, алгоритмическое дифференцирование.

## 9. Итоги лекции

1. Минимум — стационарная точка: $\nabla f(x^\ast)=0$. Обратное неверно: стационарной бывает и седло, и максимум.
2. Второй порядок: $\nabla^2f\succeq0$ необходимо, $\nabla^2f\succ0$ достаточно; между ними — зазор ($x^4$, $x^3$, $-x^4$), который закрывает только выпуклость. Для квадратичных функций ответ полный: чаша, седло, жёлоб.
3. Градиентный спуск $x_{k+1}=x_k-\alpha\nabla f(x_k)$: антиградиент — самый крутой спуск. Постоянный шаг $\alpha<2/L$ гарантирует убывание (лемма о спуске); точный шаг на квадратичной — $g^\top g/g^\top Qg$; backtracking по Армихо — рабочий рецепт без знания $L$.
4. Скорость линейная с множителем $1-1/\kappa$: итераций $\approx\kappa\ln(1/\varepsilon)$. **Число обусловленности** $\kappa=\lambda_{\max}/\lambda_{\min}$ — единственное, что определяет, ползёт метод или летит.
5. $\kappa$ — свойство координат, а не задачи: масштабирование переменных (предобуславливание) — первое лекарство. Розенброк $\kappa\approx2500$, цепь $\kappa\sim N^2$, регрессия без стандартизации $\kappa\sim10^5$.
6. Линейная, сверхлинейная, квадратичная сходимость — «сколько цифр прибавляет шаг». Ньютон: квадратичная модель, один шаг на квадратичной задаче, $7$ итераций против $13\,756$ на Розенброке — но только вблизи минимума и за цену гессиана.

## 10. Что дальше

**Лекция 5** — метод Ньютона всерьёз: локальная сходимость и теорема о сжатии, квази-ньютоновские обновления (BFGS, L-BFGS), метод Гаусса–Ньютона для задач МНК, и почему для всех них важно то, что мы сегодня увидели на Розенброке. **Лекция 6** — что делать вдали от минимума: линейный поиск (Армихо, Вольфе), доверительная область, сопряжённые градиенты. **Лекция 7** — как считать производные.

**Домашнее задание 3** выдаётся 7 октября, после лекции 5: градиентный спуск, Ньютон и BFGS — реализация и сравнение на сегодняшних задачах (см. [программу курса](../../syllabus.md#4-домашние-задания)). **Домашнее задание 2** — срок 7 октября.

**Литература к лекции.** Diehl, гл. 4 (условия оптимальности без ограничений) и §5.1 (скорости сходимости); Nocedal & Wright, §2.1 (условия оптимальности), §2.2 и §3.3 (направления спуска, скорость градиентного метода); Boyd & Vandenberghe, §9.1–9.3 (сильная выпуклость, число обусловленности, градиентный метод); Поляк, гл. 1 (градиентный метод).

## 11. Семинар: задачи у доски

Вторая половина пары, ~33 минуты, — задачи, которые решаются руками. Ноутбук [`demo04.ipynb`](demo04.ipynb) на занятии не открываем (кроме части (c) с Розенброком, если останется время): он для самостоятельной проверки тех же задач в коде после пары. Условия — в разделе 12, разбор с ответами — в [`exercises04.ipynb`](exercises04.ipynb).

| Мин | Задача | Что тренирует |
|-----|--------|---------------|
| 5 | **12.1** Стационарные точки $(x_1^2-1)^2+x_2^2$: седло и два минимума по гессиану; куда придёт спуск из $(0,0.5)$ | классификация по гессиану; седло — ловушка для спуска |
| 5 | **12.2** $x^4$, $x^3$, $-x^4$; $x_1^2\pm x_2^4$ | зазор между необходимым и достаточным |
| 10 | **12.3** Спуск на $\tfrac12(x_1^2+\kappa x_2^2)$ с постоянным шагом: диапазон $\alpha$, лучший $\alpha$, итераций на одну цифру | откуда берётся $\kappa$ — руками, покоординатно |
| 7 | **12.4** Розенброк: гессиан в $(1,1)$, $\kappa$, оценка числа итераций против демо | оценка $\kappa\ln(1/\varepsilon)$ на живых числах |
| 6 | **12.5** Километры против метров: $\kappa=10^6$ из ничего; предобусловленный шаг | масштабирование = замена переменных |

Дома: 12.6 (лемма о спуске — доказать), 12.7 (определить скорость по последовательности), 12.8 (самый крутой спуск и ортогональность), 12.9 (цепь: $\kappa\sim N^2$), 12.10 (логистическая регрессия: выпуклость и масштаб признаков); затем прогнать `demo04.ipynb`, части (a)–(e).

## 12. Упражнения

Для разбора на занятии и самостоятельно. Подробный разбор с проверкой кодом — ноутбук [`exercises04.ipynb`](exercises04.ipynb), краткие ответы — [`exercises04.md`](exercises04.md).

**12.1.** Стационарные точки $f(x)=(x_1^2-1)^2+x_2^2$ (пример лекции 2, §4). Найдите все точки с $\nabla f=0$, выпишите гессиан в каждой и классифицируйте: минимум, максимум, седло. Сверьте с картой $\lambda_{\min}\nabla^2f$ из лекции 2. Куда придёт градиентный спуск из $(0,\,0.5)$? Из $(0.01,\,0.5)$?

**12.2.** Зазор между необходимым и достаточным условием. (а) Для $x^4$, $x^3$, $-x^4$ в нуле: что говорят условия первого и второго порядка и что на самом деле? (б) То же для $f(x)=x_1^2+x_2^4$ и $g(x)=x_1^2-x_2^4$ в нуле: гессианы одинаковы, а судьбы разные. (в) Придумайте функцию двух переменных, у которой в нуле $\nabla f=0$, $\nabla^2f\succ0$ по одному направлению и $=0$ по другому, и ноль — не минимум.

**12.3.** Градиентный спуск на $f(x)=\tfrac12(x_1^2+\kappa x_2^2)$, $\kappa\ge1$, с постоянным шагом $\alpha$. Выпишите итерацию покоординатно. При каких $\alpha$ метод сходится из любой точки? Какой $\alpha$ даёт наименьший множитель сжатия $\max(|1-\alpha|,\,|1-\alpha\kappa|)$ и чему он равен? Сколько итераций нужно на одну верную десятичную цифру при $\kappa=50$ для $\alpha=1/\kappa$ и для лучшего $\alpha$? Нарисуйте первые итерации из $(\kappa,1)$ при $\alpha=1/\kappa$.

**12.4.** Розенброк $f(x)=(1-x_1)^2+100(x_2-x_1^2)^2$. Выпишите гессиан в $(1,1)$, найдите его собственные числа и $\kappa$. Оцените, сколько итераций градиентного спуска с шагом $1/\lambda_{\max}$ нужно, чтобы вблизи минимума уменьшить ошибку в $10^6$ раз. Сравните с числом итераций из демо (часть c) и с методом Ньютона.

**12.5.** Единицы измерения. Цель $f(x)=\tfrac12(x_1^2+x_2^2)$, обе координаты — расстояния в метрах. Перепишите $f$ через $u_1=x_1/1000$ (километры) и $u_2=x_2$. Чему равно $\kappa$ в переменных $u$? Сколько итераций градиентного спуска по $u$ с шагом $1/\lambda_{\max}$ нужно на одну верную цифру? Покажите, что шаг $u_{k+1}=u_k-\alpha D^{-1}\nabla_u f(u_k)$ с $D=\operatorname{diag}(10^6,1)$ — это градиентный спуск в исходных переменных $x$, и объясните, почему масштабирование переменных — первое, что стоит сделать перед запуском любого метода.

**12.6.** Лемма о спуске. Пусть $\|\nabla f(x)-\nabla f(y)\|\le L\|x-y\|$ для всех $x,y$. Докажите, что $f(y)\le f(x)+\nabla f(x)^\top(y-x)+\tfrac L2\|y-x\|^2$ (подсказка: $f(y)-f(x)=\int_0^1\nabla f(x+t(y-x))^\top(y-x)\,dt$), и выведите $f(x-\alpha\nabla f(x))\le f(x)-\alpha\big(1-\tfrac{\alpha L}2\big)\|\nabla f(x)\|^2$. При каких $\alpha$ функция гарантированно убывает? Какой $\alpha$ даёт наибольшее гарантированное убывание? Чему равно $L$ для квадратичной $\tfrac12x^\top Qx$, $Q\succeq0$?

**12.7.** Скорость по последовательности. Даны ошибки $\|e_k\|$ четырёх методов: (а) $10^{-1},10^{-2},10^{-3},10^{-4},\dots$; (б) $10^{-1},10^{-2},10^{-4},10^{-8},\dots$; (в) $1/k$; (г) $1/k!$. Определите тип сходимости каждой (линейная — с каким множителем, сверхлинейная, квадратичная, ни одна из них). Сколько итераций нужно каждой, чтобы от $10^{-1}$ дойти до $10^{-12}$?

**12.8.** Направление наискорейшего спуска. (а) Докажите, что среди единичных векторов $p$ производная по направлению $\nabla f(x)^\top p$ минимальна при $p=-\nabla f(x)/\|\nabla f(x)\|$. (б) Докажите, что $\nabla f(x)$ ортогонален линии уровня $\{y: f(y)=f(x)\}$ (подсказка: продифференцируйте $f(\gamma(t))=\mathrm{const}$ вдоль кривой $\gamma$ на линии уровня). (в) Покажите, что при точном линейном поиске $\nabla f(x_{k+1})\perp\nabla f(x_k)$ — отсюда зигзаг на картинке 04.

**12.9.** Цепь без пола (лекция 1, §7.3). Покажите, что гессиан энергии — блочно-диагональная матрица $\operatorname{diag}(DK,\,DK)$ с трёхдиагональной $K$ ($2$ на диагонали, $-1$ рядом), не зависящая от $x$. Собственные числа $K$ равны $2-2\cos\frac{j\pi}{N+1}$, $j=1,\dots,N$ (примите без доказательства). Найдите $\kappa$ как функцию $N$ и покажите, что $\kappa\approx\big(\tfrac{2(N+1)}{\pi}\big)^2$ при больших $N$. Сколько итераций градиентного спуска нужно при $N=40$, чтобы уменьшить ошибку в $10^6$ раз? Сколько шагов Ньютона? Почему здесь Ньютон — это просто `np.linalg.solve`?

**12.10.** Логистическая регрессия (лекция 1, §7.5). Для $f(w)=\frac1n\sum_{i=1}^n\log\big(1+e^{-y_i\,x_i^\top w}\big)$, $y_i\in\{-1,1\}$, найдите $\nabla f$ и $\nabla^2f$ и докажите, что $f$ выпукла (гессиан имеет вид $\frac1n X^\top SX$ с диагональной $S\succeq0$). Объясните, почему признаки разного масштаба (возраст в годах и доход в рублях) дают большое $\kappa$ и как стандартизация признаков это лечит. Проверьте в демо (часть d).

**Домашнее задание 3** — по градиентным методам, выдаётся 7 октября.

## Вопросы для самопроверки

1. Почему в локальном минимуме $\nabla f=0$, и почему из $\nabla f=0$ минимум не следует? Приведите по примеру седла и максимума.
2. В чём зазор между $\nabla^2f\succeq0$ и $\nabla^2f\succ0$? Какие три функции одной переменной его демонстрируют, и что происходит с зазором для выпуклых функций?
3. Почему антиградиент — самый крутой спуск? Что такое направление спуска вообще, и как выглядит их множество на картинке линий уровня?
4. Что гарантирует лемма о спуске и при каких шагах? Что произойдёт при $\alpha>2/L$ — на квадратичной функции и на Розенброке?
5. Что такое число обусловленности, как оно входит в число итераций градиентного спуска и почему масштабирование переменных его меняет, хотя задача та же?
6. Как по графику $\log\lVert e_k\rVert$ отличить линейную, сверхлинейную и квадратичную сходимость? Какая у градиентного спуска и какая у Ньютона?